In [1]:
from pathlib import Path
import json
import re

import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

d:\Truong_pipeline\My_Code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Resolve paths whether the notebook kernel starts in the project root or in src/.
BASE_DIR = Path.cwd().parent / "L30_data"

CAPTION_DIR = BASE_DIR / "captions"
OUTPUT_DIR = BASE_DIR / "caption_embeddings"
MODEL_NAME = "BAAI/bge-m3"
BATCH_SIZE = 64
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
def natural_sort_key(path: Path):
    return [int(part) if part.isdigit() else part.lower() for part in re.split(r"(\d+)", path.name)]


def list_caption_files():
    if not CAPTION_DIR.is_dir():
        raise FileNotFoundError(f"Caption directory not found: {CAPTION_DIR}")
    return sorted((path for path in CAPTION_DIR.iterdir() if path.suffix.lower() == ".json"), key=natural_sort_key)


def text_from_shot(shot):
    value = shot.get("caption") or shot.get("subtitle") or ""
    if value is None:
        return ""
    return str(value).strip()


def encode_texts(texts, model):
    if not texts:
        embedding_size = model.get_sentence_embedding_dimension()
        return np.empty((0, embedding_size), dtype=np.float32)

    processed_texts = [t if (t and t.strip()) else " " for t in texts]

    # model.encode đã tự động chia BATCH_SIZE
    embeddings = model.encode(
        processed_texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    return embeddings.astype(np.float32)


def embed_caption_file(caption_path, model):
    with caption_path.open("r", encoding="utf-8") as file:
        shots = json.load(file)

    if not isinstance(shots, list):
        raise ValueError(f"Invalid caption file format (expected list): {caption_path}")

    texts = [text_from_shot(shot) for shot in shots if isinstance(shot, dict)]
    if any(not text for text in texts):
        raise ValueError(f"Missing caption text in: {caption_path}")

    return encode_texts(texts, model)


print(f"Repository: {BASE_DIR}")
print(f"Captions: {CAPTION_DIR}")
print(f"Output: {OUTPUT_DIR}")
print(f"Device: {DEVICE}")

Repository: d:\Truong_pipeline\My_Code\L30_data
Captions: d:\Truong_pipeline\My_Code\L30_data\captions
Output: d:\Truong_pipeline\My_Code\L30_data\caption_embeddings
Device: cuda


In [4]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loading {MODEL_NAME} on {DEVICE}...")
model = SentenceTransformer(MODEL_NAME).to(DEVICE)
model.eval()

Loading BAAI/bge-m3 on cuda...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 24093.51it/s]


SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'XLMRobertaModel'})
  (1): Pooling({'embedding_dimension': 1024, 'pooling_mode': 'cls', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)

In [5]:
caption_files = list_caption_files()
print(f"Found {len(caption_files)} caption files")

for caption_path in tqdm(caption_files, desc="Embedding captions"):
    stem = caption_path.stem
    embedding_path = OUTPUT_DIR / f"{stem}.npy"

    if embedding_path.exists():
        print(f"Skipping existing file: {embedding_path.name}")
        continue

    embeddings = embed_caption_file(caption_path, model)
    np.save(embedding_path, embeddings)

    print(f"Saved {embedding_path.name}: shape={embeddings.shape}")

print("Done.")

Found 96 caption files


Embedding captions:   1%|          | 1/96 [00:00<01:28,  1.07it/s]

Saved L30_V001.npy: shape=(6, 1024)
Saved L30_V002.npy: shape=(7, 1024)


Embedding captions:   5%|▌         | 5/96 [00:01<00:18,  4.98it/s]

Saved L30_V003.npy: shape=(14, 1024)
Saved L30_V004.npy: shape=(4, 1024)
Saved L30_V005.npy: shape=(3, 1024)


...

Saved L30_V091.npy: shape=(11, 1024)
Saved L30_V092.npy: shape=(7, 1024)


Embedding captions:  98%|█████████▊| 94/96 [00:12<00:00,  7.42it/s]

Saved L30_V093.npy: shape=(10, 1024)
Saved L30_V094.npy: shape=(9, 1024)


Embedding captions: 100%|██████████| 96/96 [00:12<00:00,  7.69it/s]

Saved L30_V095.npy: shape=(13, 1024)
Saved L30_V096.npy: shape=(4, 1024)
Done.
